In [ ]:
"""
Web Scraper for News Articles
==============================
This script scrapes news headlines from various sources using free proxies.
Scrapes up to 100 latest records per category.

Author: [Your Name]
Roll No: [Your Roll Number]
Date: October 2025

Usage:
    python scraper.py

Note: This is a template. You need to:
1. Add specific news website URLs
2. Configure CSS selectors for headlines
3. Set up proxy rotation if needed
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from datetime import datetime

# Configuration
MAX_RECORDS = 100  # Maximum records to scrape per category
DELAY_MIN = 1  # Minimum delay between requests (seconds)
DELAY_MAX = 3  # Maximum delay between requests (seconds)

# Free proxy list (from https://www.scraperapi.com/blog/best-10-free-proxies-and-free-proxy-lists-for-web-scraping/)
FREE_PROXIES = [
    # Add proxies here, format: 'http://ip:port'
    # Example: 'http://123.456.789.012:8080'
    # You can get free proxies from:
    # - https://free-proxy-list.net/
    # - https://www.sslproxies.org/
    # - https://www.us-proxy.org/
]

# News sources to scrape (REPLACE WITH ACTUAL URLs)
NEWS_SOURCES = {
    'Politics': [
        'https://example.com/politics',  # Replace with actual URL
    ],
    'Business': [
        'https://example.com/business',  # Replace with actual URL
    ],
    'Sports': [
        'https://example.com/sports',  # Replace with actual URL
    ],
    'Entertainment': [
        'https://example.com/entertainment',  # Replace with actual URL
    ]
}

def get_random_user_agent():
    """Return a random user agent to avoid detection"""
    user_agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36',
    ]
    return random.choice(user_agents)

def get_proxy():
    """Return a random proxy from the list"""
    if FREE_PROXIES:
        return {'http': random.choice(FREE_PROXIES)}
    return None

def scrape_headlines(url, category, max_records=100):
    """
    Scrape headlines from a given URL
    
    Args:
        url: URL to scrape
        category: Category of news (Politics, Business, Sports, Entertainment)
        max_records: Maximum number of headlines to scrape
    
    Returns:
        List of tuples (headline, category)
    """
    headlines = []
    
    try:
        print(f"Scraping {category} from {url}...")
        
        # Setup headers
        headers = {
            'User-Agent': get_random_user_agent(),
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate',
            'Connection': 'keep-alive',
        }
        
        # Get proxy (optional)
        proxy = get_proxy()
        
        # Make request
        response = requests.get(
            url, 
            headers=headers, 
            proxies=proxy,
            timeout=10
        )
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # IMPORTANT: Modify these selectors based on the target website
        # Example selectors (you need to inspect the website and update these):
        headline_elements = soup.find_all('h2', class_='headline')  # Modify selector
        # Alternative: soup.find_all('a', class_='article-title')
        # Alternative: soup.select('.news-item h3')
        
        count = 0
        for element in headline_elements:
            if count >= max_records:
                break
                
            # Extract text (modify based on HTML structure)
            headline_text = element.get_text(strip=True)
            
            # Basic filtering
            if headline_text and len(headline_text) > 10:
                headlines.append((headline_text, category))
                count += 1
        
        print(f"  ✓ Scraped {count} headlines from {category}")
        
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Error scraping {url}: {e}")
    
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")
    
    return headlines

def scrape_all_categories(max_per_category=100):
    """
    Scrape headlines from all categories
    
    Args:
        max_per_category: Maximum headlines per category
    
    Returns:
        DataFrame with columns: headline, category
    """
    all_headlines = []
    
    print("=" * 70)
    print("STARTING WEB SCRAPING")
    print("=" * 70)
    print(f"Target: {max_per_category} records per category")
    print(f"Categories: {list(NEWS_SOURCES.keys())}")
    print()
    
    for category, urls in NEWS_SOURCES.items():
        category_headlines = []
        
        for url in urls:
            # Scrape from each URL
            headlines = scrape_headlines(url, category, max_per_category)
            category_headlines.extend(headlines)
            
            # Random delay to avoid rate limiting
            delay = random.uniform(DELAY_MIN, DELAY_MAX)
            print(f"  Waiting {delay:.1f}s before next request...")
            time.sleep(delay)
            
            # Stop if we have enough headlines
            if len(category_headlines) >= max_per_category:
                category_headlines = category_headlines[:max_per_category]
                break
        
        all_headlines.extend(category_headlines)
        print(f"Total {category} headlines: {len(category_headlines)}\n")
    
    # Create DataFrame
    df = pd.DataFrame(all_headlines, columns=['headline', 'category'])
    
    print("=" * 70)
    print("SCRAPING COMPLETE")
    print("=" * 70)
    print(f"Total headlines scraped: {len(df)}")
    print(f"\nCategory distribution:")
    print(df['category'].value_counts())
    
    return df

def save_data(df, filename='data/raw/news_scraped.csv'):
    """Save scraped data to CSV"""
    # Create directory if it doesn't exist
    import os
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    
    # Save to CSV
    df.to_csv(filename, index=False)
    print(f"\n✓ Data saved to: {filename}")
    
    # Save metadata
    metadata = {
        'scrape_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'total_records': len(df),
        'categories': df['category'].value_counts().to_dict()
    }
    
    metadata_df = pd.DataFrame([metadata])
    metadata_df.to_csv(filename.replace('.csv', '_metadata.csv'), index=False)
    print(f"✓ Metadata saved to: {filename.replace('.csv', '_metadata.csv')}")

def main():
    """Main function"""
    print("""
    ╔════════════════════════════════════════════════════════════════╗
    ║          NEWS ARTICLE WEB SCRAPER                              ║
    ║          Maximum 100 records per category                      ║
    ╚════════════════════════════════════════════════════════════════╝
    """)
    
    # Important warning
    print("⚠️  IMPORTANT NOTES:")
    print("   1. Update NEWS_SOURCES with actual URLs")
    print("   2. Modify CSS selectors in scrape_headlines()")
    print("   3. Respect robots.txt and website terms of service")
    print("   4. Add proxies to FREE_PROXIES if needed")
    print("   5. Some websites may block scraping\n")
    
    # Ask for confirmation
    response = input("Have you configured the script? (yes/no): ")
    if response.lower() != 'yes':
        print("\n❌ Please configure the script before running.")
        print("   Edit NEWS_SOURCES and CSS selectors in scrape_headlines()")
        return
    
    # Scrape data
    df = scrape_all_categories(max_per_category=MAX_RECORDS)
    
    # Save data
    if len(df) > 0:
        save_data(df)
    else:
        print("\n❌ No data scraped. Please check your configuration.")

if __name__ == "__main__":
    main()


"""
EXAMPLE WEBSITES TO SCRAPE (Update selectors accordingly):
==========================================================

1. Reddit News Subreddits (requires API key for better access):
   - r/politics
   - r/business
   - r/sports
   - r/entertainment

2. News Aggregators:
   - Google News (be careful with rate limits)
   - Yahoo News

3. Open News APIs (Better alternative):
   - NewsAPI.org (free tier available)
   - GNews API
   - The Guardian API

RECOMMENDED APPROACH:
====================
Instead of scraping, consider using free News APIs which are legal and reliable:

Example using NewsAPI:
```python
import requests

API_KEY = 'your_api_key'  # Get from https://newsapi.org/
url = f'https://newsapi.org/v2/top-headlines?category=business&apiKey={API_KEY}'
response = requests.get(url)
data = response.json()
headlines = [article['title'] for article in data['articles']]
```
"""